# Anime Voice Training Pipeline — Colab Notebook (T4) — **v2 Primary**

Primary Stage 2: **UniSE → VAD → ECAPA speaker verify → quality gate → [optional] per-clip ASR**.

Shares the **same Google Drive layout** as `anime_voice_training.ipynb` (legacy Aliyun):

```
MyDrive/UniSE/data/{animations,oped,reference}
MyDrive/UniSE/models/unise.ckpt   # optional fallback
MyDrive/UniSE/output              # results
```

## Before you start
1. Runtime → **T4 GPU**
2. Same data folders as the legacy notebook (do not create a new Drive tree)
3. `DASHSCOPE_API_KEY` only needed if `ENABLE_CLIP_ASR = True`
4. Do **not** upgrade `torch` / `torchaudio` on Colab T4


In [ ]:
# ============ User Configuration ============
GITHUB_REPO_URL = "https://github.com/Sufumo/Animation_audio_extractor.git"
WORK_DIR = "/content/anime_voice_training_v2"
# Shared Drive layout (same as legacy notebook)
DRIVE_DATA_DIR = "/content/drive/MyDrive/UniSE/data"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/UniSE/output"
DASHSCOPE_API_KEY = ""
HF_TOKEN = ""
# v2 options
ENABLE_CLIP_ASR = False   # True + API key → per-clip ASR (no diarization)
SPEAKER_THRESHOLD = 0.45
SPEAKER_DEVICE = "cuda"   # "cuda" on T4, "cpu" if needed
# ============================================


In [ ]:
import os
from pathlib import Path
import shutil

from google.colab import drive
drive.mount("/content/drive")

os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}

PROJECT_DIR = Path(WORK_DIR) / "repo"
if PROJECT_DIR.exists():
    !cd {PROJECT_DIR} && git pull
else:
    !git clone {GITHUB_REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
print("Project ready at:", PROJECT_DIR)
# sanity: v2 dispatcher present
assert (PROJECT_DIR / "src/pipeline/stage2_vad_verify.py").exists() or True
print("stage2_vad_verify:", (PROJECT_DIR / "src/pipeline/stage2_vad_verify.py").exists())
print("stage2_aliyun:", (PROJECT_DIR / "src/pipeline/stage2_aliyun.py").exists())


## 2. Install core dependencies


In [ ]:
%cd {PROJECT_DIR}

!grep -v -E "^(torch|torchaudio)" requirements.txt > /tmp/requirements_no_torch.txt
!pip install -q --upgrade-strategy only-if-needed -r /tmp/requirements_no_torch.txt
!pip install -q --upgrade-strategy only-if-needed speechbrain

print("Core dependencies installed.")


## 3. Clone external tool repositories


In [ ]:
%cd {WORK_DIR}

EXTERNAL_UNISE = Path(WORK_DIR) / "unified-audio" / "QuarkAudio-UniSE"
MELBAND_DIR = Path(WORK_DIR) / "Mel-Band-Roformer-Vocal-Model"

if EXTERNAL_UNISE.exists():
    !cd {EXTERNAL_UNISE} && git pull
else:
    !git clone https://github.com/alibaba/unified-audio.git

if MELBAND_DIR.exists():
    !cd {MELBAND_DIR} && git pull
else:
    !git clone https://github.com/KimberleyJensen/Mel-Band-Roformer-Vocal-Model.git {MELBAND_DIR}

print("External repos ready.")


## 4. Install tool-specific dependencies


In [ ]:
!pip install -q transformers==4.46.3
!pip install -q soundfile librosa einx x-transformers pyyaml scipy numpy
!pip install -q pytorch-lightning==2.4.0
!pip install -q einops omegaconf safetensors packaging

!grep -v -E "^(torch|torchaudio)" {MELBAND_DIR}/requirements.txt > /tmp/melband_requirements_no_torch.txt
!pip install -q --upgrade-strategy only-if-needed -r /tmp/melband_requirements_no_torch.txt

import importlib, sys
missing = []
for pkg in ["einx", "einops", "omegaconf", "x_transformers", "safetensors", "packaging"]:
    try:
        importlib.import_module(pkg)
    except Exception as e:
        missing.append((pkg, e))
if missing:
    print("Missing packages:")
    for pkg, err in missing:
        print(f"  {pkg}: {err}")
    sys.exit(1)
else:
    print("All critical imports verified.")


## 5. Download UniSE + BiCodec checkpoints


In [ ]:
from huggingface_hub import hf_hub_download

CKPT_DIR = EXTERNAL_UNISE / "checkpoints"
os.makedirs(CKPT_DIR / "BiCodec", exist_ok=True)
os.makedirs(CKPT_DIR / "wav2vec2-large-xlsr-53", exist_ok=True)

spark_files = [
    "config.yaml",
    "BiCodec/config.yaml",
    "BiCodec/model.safetensors",
    "wav2vec2-large-xlsr-53/README.md",
    "wav2vec2-large-xlsr-53/config.json",
    "wav2vec2-large-xlsr-53/preprocessor_config.json",
    "wav2vec2-large-xlsr-53/pytorch_model.bin",
]
for f in spark_files:
    print("Downloading", f)
    hf_hub_download(repo_id="SparkAudio/Spark-TTS-0.5B", filename=f, local_dir=str(CKPT_DIR))

unise_stable = CKPT_DIR / "unise.ckpt"
try:
    print("Downloading UniSE checkpoint from HuggingFace...")
    hf_hub_download(
        repo_id="QuarkAudio/QuarkAudio-UniSE",
        filename="epoch=20-step-109367.ckpt",
        local_dir=str(CKPT_DIR),
    )
    if not unise_stable.exists():
        shutil.copy(CKPT_DIR / "epoch=20-step-109367.ckpt", unise_stable)
except Exception as e:
    print("HuggingFace download failed:", e)
    drive_ckpt = Path("/content/drive/MyDrive/UniSE/models/unise.ckpt")
    if drive_ckpt.exists():
        print("Using UniSE checkpoint from Google Drive:", drive_ckpt)
        shutil.copy(str(drive_ckpt), str(unise_stable))
    else:
        raise RuntimeError(
            f"Could not download UniSE and no fallback at {drive_ckpt}."
        )

print("UniSE ready. Checkpoint:", unise_stable)


## 6. Download Mel-Band-Roformer checkpoint


In [ ]:
MELBAND_CKPT = MELBAND_DIR / "MelBandRoformer.ckpt"
if not MELBAND_CKPT.exists():
    hf_hub_download(
        repo_id="KimberleyJSN/melbandroformer",
        filename="MelBandRoformer.ckpt",
        local_dir=str(MELBAND_DIR),
    )
print("Mel-Band-Roformer ready:", MELBAND_CKPT)


## 7. Prepare input data (shared Drive paths)


In [ ]:
DATA_DIR = Path(DRIVE_DATA_DIR)
SOURCE_DIR = DATA_DIR / "animations"
OPED_DIR = DATA_DIR / "oped"
REFERENCE_DIR = DATA_DIR / "reference"

print("Using data from:", DATA_DIR)
print("animations exists:", SOURCE_DIR.exists(), "| files:", list(SOURCE_DIR.iterdir()) if SOURCE_DIR.exists() else [])
print("oped exists:", OPED_DIR.exists(), "| files:", list(OPED_DIR.iterdir()) if OPED_DIR.exists() else [])
print("reference exists:", REFERENCE_DIR.exists(), "| files:", list(REFERENCE_DIR.iterdir()) if REFERENCE_DIR.exists() else [])

TASK_DIR = Path(WORK_DIR) / "task"
TASK_DIR.mkdir(parents=True, exist_ok=True)


## 8. Write Colab config (mode=v2)


In [ ]:
import yaml

components = [
    "unise_tse_v1",
    "vad_segments",
    "speaker_verify",
    "quality_gate",
    "merge_output",
]
if ENABLE_CLIP_ASR and DASHSCOPE_API_KEY:
    components.insert(-1, "clip_asr")

cfg = {
    "source_dir": str(SOURCE_DIR),
    "oped_dir": str(OPED_DIR) if OPED_DIR.exists() else None,
    "reference_dir": str(REFERENCE_DIR),
    "task_dir": str(TASK_DIR),
    "stage1": {
        "enabled": True,
        "components": ["mp4_to_wav", "oped_removal", "bgm_removal"],
        "sample_rate": 44100,
        "mono": False,
        "bit_depth": 16,
        "bgm_segment_seconds": 360.0,
        "keep_instrumental": False,
    },
    "stage2": {
        "enabled": True,
        "mode": "v2",
        "components": components,
        "segment_seconds": 360.0,
        "clip_padding_sec": 0.1,
        "min_clip_duration_sec": 1.5,
        "max_clip_duration_sec": 12.0,
        "merge_gap_sec": 2.0,
        "vad_method": "auto",
        "vad_top_db": 35.0,
        "vad_min_silence_sec": 0.3,
        "speaker_threshold": SPEAKER_THRESHOLD,
        "speaker_backend": "ecapa",
        "speaker_device": SPEAKER_DEVICE,
        "normalize_loudness": True,
        "target_lufs": -23.0,
        "skip_asr": not (ENABLE_CLIP_ASR and bool(DASHSCOPE_API_KEY)),
    },
    "melband_roformer": {
        "project_dir": str(MELBAND_DIR),
        "model_path": str(MELBAND_CKPT),
        "config_path": None,
    },
    "unise": {
        "project_dir": str(EXTERNAL_UNISE),
        "ckpt_path": str(unise_stable),
        "accelerator": "gpu",
        "devices": 1,
    },
    "aliyun": {"dashscope_api_key": DASHSCOPE_API_KEY or None},
    "resume": True,
    "output_dir": "output",
}

COLAB_CONFIG = PROJECT_DIR / "configs" / "colab_v2.yaml"
with open(COLAB_CONFIG, "w", encoding="utf-8") as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

if DASHSCOPE_API_KEY:
    os.environ["DASHSCOPE_API_KEY"] = DASHSCOPE_API_KEY
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_TOKEN"] = HF_TOKEN

print("Config:", COLAB_CONFIG)
print("mode:", cfg["stage2"]["mode"])
print("components:", components)


## 9. Run the pipeline


In [ ]:
import os
os.environ["DASHSCOPE_API_KEY"] = DASHSCOPE_API_KEY
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_TOKEN"] = HF_TOKEN

%cd {PROJECT_DIR}
print("Running v2 pipeline...")

oped_args = f"--oped-dir {OPED_DIR}" if OPED_DIR.exists() else ""
!python {PROJECT_DIR}/main.py     --config {COLAB_CONFIG}     --task-dir {TASK_DIR}     --source-dir {SOURCE_DIR}     --reference-dir {REFERENCE_DIR}     {oped_args}     --stage all     --mode v2     --resume


## 10. Copy results to shared Drive output


In [ ]:
import shutil
from pathlib import Path

RESULT_DIR = TASK_DIR / "output" / "stage2"
DRIVE_RESULT = Path(DRIVE_OUTPUT_DIR)

if RESULT_DIR.exists():
    DRIVE_RESULT.mkdir(parents=True, exist_ok=True)
    for src in RESULT_DIR.rglob("*"):
        if src.is_file():
            dst = DRIVE_RESULT / src.relative_to(RESULT_DIR)
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(src), str(dst))
    print("Results copied to Google Drive:", DRIVE_RESULT)
    # quick summary
    for ep in sorted(RESULT_DIR.glob("episode_*")):
        idx = ep.name.split("_")[-1]
        scores = ep / f"ep{idx}_speaker_scores.json"
        summary = ep / f"ep{idx}_summary.json"
        print("---", ep.name)
        if summary.exists():
            print(summary.read_text()[:400])
        if scores.exists():
            import json
            data = json.loads(scores.read_text())
            print(f"accepted={data.get('num_accepted')} rejected={data.get('num_rejected')} backend={data.get('backend')}")
else:
    print("No Stage 2 output found:", TASK_DIR)
